In [2]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.messages import HumanMessage,AIMessage,SystemMessage

load_dotenv()

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

In [3]:
llm=ChatGroq(model='openai/gpt-oss-120b')

In [4]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D231B771A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D231BF7110>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
# CineSage Analytics is a media intelligence
# company that helps streaming platforms and
# production houses organize and analyze large
# volumes of movie data.

# Every day, the company receives:
# · Movie descriptions
# · Press releases
# . Blog articles
# · Review summaries
# · Metadata from multiple sources

In [6]:
# CineSage wants to build an Al-powered tool
# that:
# 1. Takes a raw paragraph about a movie
# 2. Extracts important structured information
# 3. Generates a clean summary
# 4. Returns it in JSON format
# 5. Stores it in their database

In [7]:
from pydantic import BaseModel,Field

from typing import Optional

class MovieInfo(BaseModel):
    movie_name: Optional[str] = Field(description="Name of the movie")
    director: Optional[str] = Field(description="Movie director")
    release_year: Optional[int] = Field(description="Release year")
    genre: Optional[list[str]] = Field(description="Movie genres")
    lead_actors: Optional[list[str]] = Field(description="Lead actors")
    summary: str = Field(description="Summary based only on the given paragraph")
    missing_information: list[str] = Field(
        description="Requested details not provided in the paragraph"
    )



In [8]:

from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
("system", """You are an expert Movie Information Extraction Agent .

Your task is to read the movie paragraph provided by the user
and extract all relevant movie information, and summarize the paragraph.

Extract the following details:
1. movie_name
2. director
3. release_year
4. genre
5. lead_actors
6. give summary of the info


RULES:
- Extract information only from the given paragraph.
- Do not use external knowledge.
- Never invent missing information.
- if information is missing then ask for it but just dont invent anything on your own...
"""),
        ("human", "{para}")
])

user_input=input("enter para: ")

llm_with_structure_op=llm.with_structured_output(MovieInfo)
response = (prompt | llm_with_structure_op).invoke({"para": user_input})

print(response)


movie_name='Inception' director='Christopher Nolan' release_year=2010 genre=['science-fiction', 'action'] lead_actors=['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Elliot Page', 'Tom Hardy', 'Ken Watanabe', 'Marion Cotillard'] summary="Inception (2010) is a science‑fiction action film directed by Christopher Nolan, starring Leonardo DiCaprio as Dom Cobb with supporting performances by Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, and Marion Cotillard. The plot follows Cobb and his team as they attempt to plant an idea in a target's mind through the process of inception, exploring themes of dreams, reality, memory, and the subconscious." missing_information=[]


In [9]:
type(response)

__main__.MovieInfo

In [10]:
from langchain_core.output_parsers import JsonOutputParser

from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
("system", """You are an expert Movie Information Extraction Agent .

Your task is to read the movie paragraph provided by the user
and extract all relevant movie information, and summarize the paragraph.

Extract the following details:
1. movie_name
2. director
3. release_year
4. genre
5. lead_actors
6. give summary of the info


RULES:
- Extract information only from the given paragraph.
- Do not use external knowledge.
- Never invent missing information.
- if information is missing then ask for it but just dont invent anything on your own...
- use the attachedd formated strucute 

{format_instructions}
"""),
        ("human", "{para}")
])


parser = JsonOutputParser()
prompt = prompt.partial(
    format_instructions=parser.get_format_instructions()
)
res=(prompt|llm|parser).invoke({"para":user_input})
res

{'movie_name': 'Inception',
 'director': 'Christopher Nolan',
 'release_year': 2010,
 'genre': 'science-fiction action',
 'lead_actors': ['Leonardo DiCaprio',
  'Joseph Gordon-Levitt',
  'Elliot Page',
  'Tom Hardy',
  'Ken Watanabe',
  'Marion Cotillard'],
 'summary': "Inception (2010) is a science‑fiction action film written and directed by Christopher Nolan. It follows Dom Cobb (Leonardo DiCaprio), a thief who enters people's dreams to steal secrets, and his team—including Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, and Marion Cotillard—as they attempt to plant an idea in a target's mind through the process of inception, exploring themes of dreams, reality, memory, and the subconscious."}

In [11]:
import sqlite3

conn=sqlite3.connect(r'E:\chatbot\cineSage\cinesage.db')
cursor=conn.cursor()

cursor.execute("""
        CREATE TABLE IF NOT EXISTS movies (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            movie_name TEXT,
            director TEXT,
            release_year INTEGER,
            genre TEXT,
            lead_actors TEXT,
            summary TEXT
        )
    """)
conn.commit()
print("Movies table created successfully!")
conn.close()



Movies table created successfully!


In [15]:
res['movie_name']

'Inception'

In [16]:
res

{'movie_name': 'Inception',
 'director': 'Christopher Nolan',
 'release_year': 2010,
 'genre': 'science-fiction action',
 'lead_actors': ['Leonardo DiCaprio',
  'Joseph Gordon-Levitt',
  'Elliot Page',
  'Tom Hardy',
  'Ken Watanabe',
  'Marion Cotillard'],
 'summary': "Inception (2010) is a science‑fiction action film written and directed by Christopher Nolan. It follows Dom Cobb (Leonardo DiCaprio), a thief who enters people's dreams to steal secrets, and his team—including Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, and Marion Cotillard—as they attempt to plant an idea in a target's mind through the process of inception, exploring themes of dreams, reality, memory, and the subconscious."}

In [17]:
import json
conn=sqlite3.connect(r'E:\chatbot\cineSage\cinesage.db')
cursor=conn.cursor()

cursor.execute("""
        INSERT INTO movies (
            movie_name,
            director,
            release_year,
            genre,
            lead_actors,
            summary
        ) VALUES (?,?,?,?,?,?)
    """,(
        res['movie_name'],
        res['director'],
        res['release_year'],
        res['genre'],
        json.dumps(res['lead_actors']),
        res['summary']
    ))
conn.commit()
print("records inserted successfully!")
conn.close()

records inserted successfully!
